# Deploy the deep research backend

**This notebook is optional.** Skip it if the core gateway deployment was run from this version of
the repo - its Bicep already deploys the Norway East research hub, the `deep-research` model
deployment and the APIM routing for it.

Run this notebook only if:
- the core gateway deployment was run from an older version, which deployed `o3-deep-research`
  (retiring 2026-11-19) or no research hub at all, OR
- You want a dedicated APIM subscription key for the deep research workload.

## What this notebook does

1. Checks whether the `deep-research` deployment and the Responses routing policy already exist.
2. If not: deploys `main.bicep` to the core resource group, creating or updating:
   - `aif-research-{suffix}` Norway East CognitiveServices account
   - `deep-research` model deployment (`gpt-5.6-sol`)
   - `openai-research` APIM backend + `chat-research` routing policy
   - a policy on the `responses` operation that sends `model: deep-research` to the research hub
   - `foundry-gateway-dr` APIM subscription
3. Writes `DR_MODEL` and `DR_GATEWAY_KEY` to `.env`.

An existing `o3-deep-research` deployment is left in place. It stops answering when the model
retires, and nothing in the repo calls it after this notebook runs.

## Prerequisites

- Core gateway deployed (core APIM, `GATEWAY_URL` in `.env`)
- `az login` done

## Step 1: Load configuration

In [1]:
import json
import os
import subprocess
import base64
from pathlib import Path

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
env_file = repo_root / '.env'

with open(env_file) as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

GATEWAY_URL = os.environ['GATEWAY_URL']
CHAT_MODEL  = os.environ.get('CHAT_MODEL', 'gpt-5.4-mini')

print(f'Gateway URL  : {GATEWAY_URL}')
print(f'Chat model   : {CHAT_MODEL}')

Gateway URL  : https://apim-foundry-c2676f.azure-api.net/openai
Chat model   : gpt-4.1-mini


## Step 2: Resolve core resource group and principal

In [2]:
# Derive APIM name and suffix from GATEWAY_URL
# e.g. https://apim-foundry-{suffix}.azure-api.net/openai -> apim-foundry-{suffix}
APIM_NAME = GATEWAY_URL.split('//')[1].split('.')[0]
SUFFIX    = APIM_NAME.split('-')[-1]          # e.g. {suffix}
CORE_RG   = f'rg-foundry-core-{SUFFIX}'

# Subscription ID + ARM base URI used by Steps 3 and 5 to call APIM management
# endpoints directly via `az rest`. This avoids requiring the `az apim` CLI
# extension, which is not bundled with the base Azure CLI.
SUB_ID = subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()
APIM_BASE_URI = (
    f'https://management.azure.com/subscriptions/{SUB_ID}'
    f'/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
)

# Get deployer principal ID from cached JWT
token   = subprocess.run(
    'az account get-access-token --query accessToken -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()
padding = '=' * (4 - len(token.split('.')[1]) % 4)
PRINCIPAL_ID = json.loads(base64.b64decode(token.split('.')[1] + padding))['oid']

print(f'APIM service : {APIM_NAME}')
print(f'Core RG      : {CORE_RG}')
print(f'Subscription : {SUB_ID}')
print(f'Principal ID : {PRINCIPAL_ID}')

APIM service : apim-foundry-c2676f
Core RG      : rg-foundry-core-c2676f
Subscription : 00000000-0000-0000-0000-000000000000
Principal ID : 00000000-0000-0000-0000-000000000000


## Step 3: Check whether the deep-research backend already exists

In [3]:
DR_MODEL     = 'deep-research'
RESEARCH_HUB = f'aif-research-{SUFFIX}'

# The backend alone is not enough: environments deployed before the model swap have the
# openai-research backend but only an o3-deep-research deployment and no Responses routing.
deployment = subprocess.run(
    f'az cognitiveservices account deployment show -g {CORE_RG} -n {RESEARCH_HUB}'
    f' --deployment-name {DR_MODEL} -o none',
    shell=True, capture_output=True, text=True
)
responses_policy = subprocess.run(
    f'az rest --method GET'
    f' --uri "{APIM_BASE_URI}/apis/openai/operations/responses/policies/policy?api-version=2024-06-01-preview"'
    f' -o none',
    shell=True, capture_output=True, text=True
)
BACKEND_READY = deployment.returncode == 0 and responses_policy.returncode == 0

if BACKEND_READY:
    print(f'✅ {DR_MODEL} deployment and Responses routing already exist - skipping Bicep deployment.')
    print('   Proceeding to read existing resources.')
else:
    print(f'ℹ️  {DR_MODEL} deployment or Responses routing not found - will deploy main.bicep.')

ℹ️  deep-research deployment or Responses routing not found - will deploy main.bicep.


## Step 4: Deploy main.bicep (if needed)

Runs only when the deployment or the Responses routing is missing. Takes ~5 minutes.

In [4]:
if not BACKEND_READY:
    result = subprocess.run(
        [
            'az', 'deployment', 'group', 'create',
            '-g', CORE_RG,
            '--template-file', 'main.bicep',
            '-p', f'deployerPrincipalId={PRINCIPAL_ID}',
            '-p', f'existingApimName={APIM_NAME}',
            '-p', f'suffix={SUFFIX}',
            '--name', 'dr-backend',
            '-o', 'table',
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Bicep deployment failed - see stderr above')
    print('✅ Bicep deployment complete')
else:
    print('ℹ️  Skipped (backend already deployed)')

Name        State      Timestamp                         Mode         ResourceGroup
----------  ---------  --------------------------------  -----------  ----------------------
dr-backend  Succeeded  2026-10-09T09:33:30.534460+00:00  Incremental  rg-foundry-core-c2676f

✅ Bicep deployment complete


## Step 5: Resolve DR APIM subscription key

In [5]:
# Try the dedicated deep research subscription first; fall back to the alpha
# subscription if it doesn't exist yet. Uses `az rest` against ARM directly,
# matching the pattern in 10-01 and 11-01, so the `az apim` extension is not needed.
def _list_apim_subscription_key(sub_name: str) -> str | None:
    r = subprocess.run(
        f'az rest --method POST'
        f' --uri "{APIM_BASE_URI}/subscriptions/{sub_name}/listSecrets?api-version=2024-06-01-preview"'
        f' --query primaryKey -o tsv',
        shell=True, capture_output=True, text=True
    )
    return r.stdout.strip() if r.returncode == 0 and r.stdout.strip() else None

DR_GATEWAY_KEY = _list_apim_subscription_key('foundry-gateway-dr')
if DR_GATEWAY_KEY:
    print('✅ Using foundry-gateway-dr subscription key')
else:
    DR_GATEWAY_KEY = _list_apim_subscription_key('foundry-gateway-alpha')
    if DR_GATEWAY_KEY:
        print('ℹ️  Using foundry-gateway-alpha subscription key (fallback)')
    else:
        raise RuntimeError('Could not retrieve APIM subscription key. Check az login and core RG.')

print(f'DR model     : {DR_MODEL}')
print(f'DR key       : {DR_GATEWAY_KEY[:4]}... (hidden)')

✅ Using foundry-gateway-dr subscription key
DR model     : deep-research
DR key       : 805c... (hidden)


## Step 6: Write DR_* env vars to .env

In [6]:
lines = env_file.read_text().splitlines() if env_file.exists() else []

updates = {
    'DR_MODEL':       DR_MODEL,
    'DR_GATEWAY_KEY': DR_GATEWAY_KEY,
}

# Remove existing DR_* lines, then append updated values
kept = [ln for ln in lines if not any(ln.startswith(k + '=') for k in updates)]
kept += [f'{k}={v}' for k, v in updates.items()]

env_file.write_text('\n'.join(kept) + '\n')

print('✅ .env updated:')
for k, v in updates.items():
    masked = v[:4] + '...' if len(v) > 4 and 'KEY' in k else v
    print(f'   {k}={masked}')

✅ .env updated:
   DR_MODEL=deep-research
   DR_GATEWAY_KEY=805c...


## Next step

Open [`12-02-deep-research-loop.ipynb`](12-02-deep-research-loop.ipynb) to run the
agentic deep research loop over the `arxiv-nlp` knowledge base.